In [1]:
"""
.env에 저장된 KAMIS / 참가격(price.go.kr) API 키가 정상 동작하는지 확인하는 스크립트.

사전 준비:
    pip install python-dotenv requests
    프로젝트 루트에 .env 파일을 만들고 .env.example을 참고해 실제 키 값을 채워 넣을 것.

실행:
    python test_api_keys.py
"""

import os
import sys
from datetime import datetime, timedelta

import requests
from dotenv import load_dotenv

load_dotenv()  # .env 파일을 읽어 os.environ에 반영

KAMIS_CERT_KEY = os.getenv("KAMIS_CERT_KEY")
KAMIS_CERT_ID = os.getenv("KAMIS_CERT_ID")
PRICE_GOKR_SERVICE_KEY = os.getenv("PRICE_GOKR_SERVICE_KEY")


def check_env_vars() -> bool:
    """필요한 환경변수가 모두 채워져 있는지 먼저 확인."""
    missing = [
        name
        for name, value in [
            ("KAMIS_CERT_KEY", KAMIS_CERT_KEY),
            ("KAMIS_CERT_ID", KAMIS_CERT_ID),
            ("PRICE_GOKR_SERVICE_KEY", PRICE_GOKR_SERVICE_KEY),
        ]
        if not value
    ]
    if missing:
        print("[환경변수 누락] .env에 다음 값이 비어 있습니다:", ", ".join(missing))
        print("   .env.example을 복사해 .env로 만들고 실제 키 값을 채워주세요.\n")
        return False
    return True


def test_kamis() -> bool:
    """KAMIS 농수산물 가격 API 연결 테스트 (배추 등 채소류 조회)."""
    url = "https://www.kamis.or.kr/service/price/xml.do"
    yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
    params = {
        "action": "dailyPriceByCategoryList",
        "p_product_cls_code": "01",
        "p_category_code": "200",
        "p_regday": yesterday,
        "p_convert_kg_yn": "N",
        "p_cert_key": KAMIS_CERT_KEY,
        "p_cert_id": KAMIS_CERT_ID,
        "p_returntype": "json",
    }

    print("── KAMIS (농수산물 가격) ──")
    try:
        res = requests.get(url, params=params, timeout=10)
        print(f"Status Code: {res.status_code}")

        data = res.json()
        error_code = data.get("data", {}).get("error_code")
        items = data.get("data", {}).get("item", [])

        if error_code == "000" and items:
            first = items[0]
            print(f"성공 — 예: {first.get('item_name')} 당일가 {first.get('dpr1')}원")
            return True
        else:
            print(f"실패 — error_code={error_code}, 응답 일부: {res.text[:200]}")
            return False
    except Exception as e:
        print(f"요청 실패: {e}")
        return False
    finally:
        print()


def test_price_gokr() -> bool:
    """참가격(생필품 가격 정보) API 연결 테스트."""
    url = "http://openapi.price.go.kr/openApiImpl/ProductPriceInfoService/getProductInfoSvc.do"
    params = {"ServiceKey": PRICE_GOKR_SERVICE_KEY}

    print("── 참가격 (생필품 가격 정보) ──")
    try:
        res = requests.get(url, params=params, timeout=10)
        print(f"Status Code: {res.status_code}")
        text = res.text.strip()

        if text.startswith("90") or "Invalid Authentication" in text:
            print(f"실패 — 인증 오류: {text[:200]}")
            return False
        elif res.status_code == 200 and text:
            print(f"성공 — 응답 일부: {text[:200]}")
            return True
        else:
            print(f"실패 — 응답 일부: {text[:200]}")
            return False
    except Exception as e:
        print(f"요청 실패: {e}")
        return False
    finally:
        print()


def main():
    if not check_env_vars():
        sys.exit(1)

    kamis_ok = test_kamis()
    price_ok = test_price_gokr()

    print("── 결과 요약 ──")
    print(f"KAMIS       : {'OK' if kamis_ok else 'FAIL'}")
    print(f"참가격      : {'OK' if price_ok else 'FAIL'}")

    if not (kamis_ok and price_ok):
        sys.exit(1)


if __name__ == "__main__":
    main()

── KAMIS (농수산물 가격) ──
Status Code: 200
요청 실패: 'list' object has no attribute 'get'

── 참가격 (생필품 가격 정보) ──
Status Code: 200
성공 — 응답 일부: <?xml version="1.0" encoding="UTF-8"?><response>
  <result>
    <item>
      <goodId>35</goodId>
      <goodName>해표 꽃소금(1kg)</goodName>
      <productEntpCode>484</productEntpCode>
      <goodUnitDivC

── 결과 요약 ──
KAMIS       : FAIL
참가격      : OK


SystemExit: 1

c:\Users\kjw02\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
